# Part 4 — Neural Network, Ablation, XAI & Inference

**Role: Person 4 — Zeina**

This notebook is designed to work for **both Kaggle submission and the GitHub repository**.

It covers:
- shallow feed-forward neural network
- multiple NN trials
- ROC-AUC, PR-AUC, and F1 evaluation
- feature-group ablation
- permutation importance
- global SHAP
- local SHAP
- inference on complete and partially-missing records

> The final run must use the **same leakage-safe modeling table, feature set, and chronological split** used by the team's statistical models.


## 1. Imports and reproducibility

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_curve, roc_curve, confusion_matrix,
    classification_report
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

try:
    import shap
    SHAP_AVAILABLE = True
    print("SHAP:", shap.__version__)
except Exception as e:
    SHAP_AVAILABLE = False
    print("SHAP import failed:", e)


## 2. Locate the final modeling table

Expected handoff from Person 2: a CSV/Parquet file containing **one row per (raceId, driverId)** and only leakage-safe pre-race features.

This cell searches common Kaggle and local/GitHub-style paths.


In [ ]:
CANDIDATE_NAMES = [
    "modeling_table.csv",
    "modelling_table.csv",
    "final_modeling_table.csv",
    "final_modelling_table.csv",
    "model_table.csv",
    "features.csv",
]

SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/working",
    ".",
    "./data",
    "./data/processed",
]

def find_modeling_table():
    hits = []
    for root in SEARCH_ROOTS:
        if not os.path.exists(root) or os.path.isfile(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for f in filenames:
                low = f.lower()
                if low in CANDIDATE_NAMES or (
                    ("model" in low or "feature" in low)
                    and low.endswith((".csv", ".parquet"))
                ):
                    hits.append(os.path.join(dirpath, f))
    return sorted(set(hits))

hits = find_modeling_table()

print("Candidate modeling tables:")
for h in hits[:20]:
    print(" -", h)

MODEL_TABLE_PATH = hits[0] if hits else None

if MODEL_TABLE_PATH is None:
    DATA_READY = False
    df = None
    print("\nNo final modeling table found yet.")
    print("Add/attach Person 2's final modeling table, then Run All.")
else:
    if MODEL_TABLE_PATH.endswith(".parquet"):
        df = pd.read_parquet(MODEL_TABLE_PATH)
    else:
        df = pd.read_csv(MODEL_TABLE_PATH)
    DATA_READY = True
    print("\nLoaded:", MODEL_TABLE_PATH)
    print("Shape:", df.shape)
    display(df.head())


## 3. Schema checks and target configuration

In [ ]:
TARGET_CANDIDATES = ["podium", "is_podium", "target", "podium_finish"]
YEAR_CANDIDATES = ["year", "season"]

if DATA_READY:
    target_col = next((c for c in TARGET_CANDIDATES if c in df.columns), None)

    if target_col is None and "positionOrder" in df.columns:
        df["podium"] = (pd.to_numeric(df["positionOrder"], errors="coerce") <= 3).astype(int)
        target_col = "podium"

    if target_col is None:
        raise ValueError("Could not identify podium target column.")

    year_col = next((c for c in YEAR_CANDIDATES if c in df.columns), None)
    if year_col is None:
        raise ValueError("No year/season column found.")

    print("Target column:", target_col)
    print("Year column:", year_col)
    display(df[target_col].value_counts(dropna=False).rename("count").to_frame())
    print("Podium rate:", round(df[target_col].mean(), 4))

    if {"raceId", "driverId"}.issubset(df.columns):
        dup_count = df.duplicated(["raceId", "driverId"]).sum()
        print("Duplicate (raceId, driverId) rows:", dup_count)
        if dup_count:
            print("WARNING: final modeling grain is not unique yet.")


## 4. Feature selection and leakage guard

Post-race fields are excluded from model inputs.
The final feature list should be checked against Person 3's statistical models before submission.


In [ ]:
POST_RACE_LEAKAGE = {
    "position", "positionText", "positionOrder", "points", "laps",
    "milliseconds", "fastestLap", "rank", "fastestLapTime",
    "fastestLapSpeed", "statusId", "status", "resultId"
}

IDENTIFIER_ONLY = {
    "raceId", "driverId", "constructorId", "circuitId",
    "qualifyId", "driverStandingsId", "constructorStandingsId"
}

NON_FEATURE_META = {
    "date", "time", "url", "name", "forename", "surname",
    "dob", "driverRef", "constructorRef", "circuitRef"
}

if DATA_READY:
    excluded = {target_col, year_col} | POST_RACE_LEAKAGE | IDENTIFIER_ONLY | NON_FEATURE_META
    feature_cols = [c for c in df.columns if c not in excluded and df[c].notna().any()]

    print("Candidate feature count:", len(feature_cols))
    print(feature_cols)

    suspicious = [c for c in feature_cols if c in POST_RACE_LEAKAGE]
    assert not suspicious, f"Leakage features still present: {suspicious}"


## 5. Chronological split

Recommended split:
- Train: <= 2019
- Validation: 2020–2021
- Test: >= 2022


In [ ]:
if DATA_READY:
    work = df.dropna(subset=[target_col, year_col]).copy()
    work[year_col] = pd.to_numeric(work[year_col], errors="coerce")
    work = work.dropna(subset=[year_col])

    train_df = work[work[year_col] <= 2019].copy()
    val_df   = work[work[year_col].between(2020, 2021)].copy()
    test_df  = work[work[year_col] >= 2022].copy()

    print("Train:", train_df.shape)
    print("Validation:", val_df.shape)
    print("Test:", test_df.shape)

    assert len(train_df) and len(val_df) and len(test_df), "One chronological split is empty."

    X_train_raw = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(int).to_numpy()

    X_val_raw = val_df[feature_cols].copy()
    y_val = val_df[target_col].astype(int).to_numpy()

    X_test_raw = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(int).to_numpy()


## 6. Preprocessing

In [ ]:
def build_preprocessor(X):
    numeric_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    transformers = [("num", num_pipe, numeric_cols)]
    if categorical_cols:
        transformers.append(("cat", cat_pipe, categorical_cols))

    prep = ColumnTransformer(transformers, remainder="drop")
    return prep, numeric_cols, categorical_cols

if DATA_READY:
    preprocessor, numeric_cols, categorical_cols = build_preprocessor(X_train_raw)

    X_train = np.asarray(preprocessor.fit_transform(X_train_raw), dtype=np.float32)
    X_val   = np.asarray(preprocessor.transform(X_val_raw), dtype=np.float32)
    X_test  = np.asarray(preprocessor.transform(X_test_raw), dtype=np.float32)

    try:
        transformed_feature_names = preprocessor.get_feature_names_out().tolist()
    except Exception:
        transformed_feature_names = [f"x{i}" for i in range(X_train.shape[1])]

    print("Numeric raw features:", len(numeric_cols))
    print("Categorical raw features:", len(categorical_cols))
    print("Transformed feature count:", X_train.shape[1])


## 7. Evaluation utilities

In [ ]:
def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5
    f1s = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[int(np.nanargmax(f1s))])

def metric_dict(y_true, y_prob, threshold):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "Threshold": threshold,
    }

def plot_roc_pr(y_true, y_prob, title_prefix="Model"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_true, y_prob):.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix} — ROC Curve")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AP={average_precision_score(y_true, y_prob):.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix} — Precision–Recall Curve")
    plt.legend()
    plt.show()


## 8. Shallow NN architectures

In [ ]:
def build_nn_trial_1(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, activation="relu"),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy")
    return model

def build_nn_trial_2(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        Dropout(0.20),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy")
    return model

def train_nn(builder, X_train, y_train, X_val, y_val, verbose=0):
    model = builder(X_train.shape[1])
    callback = EarlyStopping(
        monitor="val_loss",
        patience=8,
        min_delta=1e-4,
        restore_best_weights=True,
    )
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=150,
        batch_size=32,
        callbacks=[callback],
        verbose=verbose,
    )
    return model, history

def plot_history(history, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history.history["loss"], label="Train loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Binary cross-entropy")
    plt.title(title)
    plt.legend()
    plt.show()


## 9. Train and compare NN trials

In [ ]:
if DATA_READY:
    model1, history1 = train_nn(build_nn_trial_1, X_train, y_train, X_val, y_val)
    model2, history2 = train_nn(build_nn_trial_2, X_train, y_train, X_val, y_val)

    plot_history(history1, "NN Trial 1 — Training History")
    plot_history(history2, "NN Trial 2 — Training History")

    val_prob_1 = model1.predict(X_val, verbose=0).ravel()
    val_prob_2 = model2.predict(X_val, verbose=0).ravel()

    thr1 = best_f1_threshold(y_val, val_prob_1)
    thr2 = best_f1_threshold(y_val, val_prob_2)

    r1 = metric_dict(y_val, val_prob_1, thr1)
    r2 = metric_dict(y_val, val_prob_2, thr2)

    nn_comparison = pd.DataFrame([
        {"Model": "NN Trial 1 (32-16)", **r1},
        {"Model": "NN Trial 2 (64-32 + dropout)", **r2},
    ]).sort_values(["PR-AUC", "F1"], ascending=False)

    display(nn_comparison)

    winner = nn_comparison.iloc[0]["Model"]
    if winner.startswith("NN Trial 1"):
        best_model, best_history, best_threshold = model1, history1, thr1
        best_label = "NN Trial 1"
    else:
        best_model, best_history, best_threshold = model2, history2, thr2
        best_label = "NN Trial 2"

    print("Selected using validation data only:", best_label)
    print("Chosen threshold:", round(best_threshold, 4))

    plot_roc_pr(y_val, best_model.predict(X_val, verbose=0).ravel(), f"{best_label} Validation")


## 10. Train / validation / unseen test results

In [ ]:
if DATA_READY:
    split_rows = []
    for split_name, Xs, ys in [
        ("Train", X_train, y_train),
        ("Validation", X_val, y_val),
        ("Test", X_test, y_test),
    ]:
        probs = best_model.predict(Xs, verbose=0).ravel()
        split_rows.append({"Split": split_name, **metric_dict(ys, probs, best_threshold)})

    final_nn_results = pd.DataFrame(split_rows)
    display(final_nn_results)

    test_prob = best_model.predict(X_test, verbose=0).ravel()
    test_pred = (test_prob >= best_threshold).astype(int)

    print(classification_report(y_test, test_pred, digits=4))
    display(pd.DataFrame(
        confusion_matrix(y_test, test_pred),
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"]
    ))

    plot_roc_pr(y_test, test_prob, f"{best_label} Unseen Test")


## 11. Feature-group ablation

In [ ]:
def infer_feature_groups(columns):
    groups = {
        "grid": [],
        "qualifying": [],
        "driver_form": [],
        "constructor_form": [],
        "standings": [],
        "circuit_context": [],
    }

    for c in columns:
        low = c.lower()

        if "grid" in low or "start" in low:
            groups["grid"].append(c)

        if "quali" in low or low.startswith("q1") or low.startswith("q2") or low.startswith("q3"):
            groups["qualifying"].append(c)

        if "driver" in low and any(k in low for k in ["form", "recent", "rolling", "podium", "finish", "experience"]):
            groups["driver_form"].append(c)

        if "constructor" in low and any(k in low for k in ["form", "recent", "rolling", "podium", "finish", "reliab"]):
            groups["constructor_form"].append(c)

        if "standing" in low or "championship" in low:
            groups["standings"].append(c)

        if any(k in low for k in ["circuit", "home", "country", "era"]):
            groups["circuit_context"].append(c)

    return {k: sorted(set(v)) for k, v in groups.items() if v}

if DATA_READY:
    feature_groups = infer_feature_groups(feature_cols)

    print("Detected feature groups:")
    for g, cols in feature_groups.items():
        print(f"{g}: {cols}")


In [ ]:
def run_ablation(raw_train, y_train, raw_val, y_val, base_features, groups, selected_builder):
    rows = []

    def fit_eval(cols, label):
        prep, _, _ = build_preprocessor(raw_train[cols])
        Xt = np.asarray(prep.fit_transform(raw_train[cols]), dtype=np.float32)
        Xv = np.asarray(prep.transform(raw_val[cols]), dtype=np.float32)

        model, _ = train_nn(selected_builder, Xt, y_train, Xv, y_val, verbose=0)
        prob = model.predict(Xv, verbose=0).ravel()
        threshold = best_f1_threshold(y_val, prob)
        m = metric_dict(y_val, prob, threshold)

        rows.append({
            "Experiment": label,
            "Num raw features": len(cols),
            **m
        })
        return m

    full = fit_eval(base_features, "Full feature set")

    for group_name, group_cols in groups.items():
        remaining = [c for c in base_features if c not in set(group_cols)]
        if remaining and group_cols:
            fit_eval(remaining, f"Without {group_name}")

    out = pd.DataFrame(rows)
    out["Δ PR-AUC vs full"] = out["PR-AUC"] - full["PR-AUC"]
    out["Δ ROC-AUC vs full"] = out["ROC-AUC"] - full["ROC-AUC"]
    out["Δ F1 vs full"] = out["F1"] - full["F1"]
    return out

if DATA_READY:
    selected_builder = build_nn_trial_1 if best_label == "NN Trial 1" else build_nn_trial_2

    ablation_results = run_ablation(
        X_train_raw, y_train,
        X_val_raw, y_val,
        feature_cols,
        feature_groups,
        selected_builder
    )

    display(ablation_results.sort_values("PR-AUC", ascending=False))

    plot_df = ablation_results.set_index("Experiment")["PR-AUC"].sort_values()
    plt.figure(figsize=(9, 5))
    plot_df.plot(kind="barh")
    plt.xlabel("Validation PR-AUC")
    plt.title("Feature-Group Ablation")
    plt.tight_layout()
    plt.show()


## 12. Permutation importance

Importance here means **model reliance**, not causality.


In [ ]:
def permutation_importance_pr_auc(model, X, y, feature_names, repeats=3, seed=SEED):
    rng = np.random.default_rng(seed)
    baseline_prob = model.predict(X, verbose=0).ravel()
    baseline = average_precision_score(y, baseline_prob)

    rows = []
    for j, name in enumerate(feature_names):
        drops = []

        for _ in range(repeats):
            Xp = X.copy()
            rng.shuffle(Xp[:, j])

            score = average_precision_score(
                y,
                model.predict(Xp, verbose=0).ravel()
            )
            drops.append(baseline - score)

        rows.append({
            "Feature": name,
            "Importance (PR-AUC drop)": float(np.mean(drops)),
            "Std": float(np.std(drops)),
        })

    return pd.DataFrame(rows).sort_values(
        "Importance (PR-AUC drop)", ascending=False
    )

if DATA_READY:
    permutation_df = permutation_importance_pr_auc(
        best_model, X_val, y_val,
        transformed_feature_names,
        repeats=3
    )

    display(permutation_df.head(20))

    top_perm = permutation_df.head(20).sort_values("Importance (PR-AUC drop)")
    plt.figure(figsize=(9, 7))
    plt.barh(top_perm["Feature"], top_perm["Importance (PR-AUC drop)"])
    plt.xlabel("Decrease in validation PR-AUC after shuffling")
    plt.title("Permutation Importance — Top 20")
    plt.tight_layout()
    plt.show()


## 13. Global SHAP

In [ ]:
if DATA_READY and SHAP_AVAILABLE:
    rng = np.random.default_rng(SEED)

    bg_n = min(50, len(X_train))
    ex_n = min(60, len(X_val))

    bg_idx = rng.choice(len(X_train), size=bg_n, replace=False)
    ex_idx = rng.choice(len(X_val), size=ex_n, replace=False)

    X_bg = X_train[bg_idx]
    X_explain = X_val[ex_idx]

    def predict_fn(x):
        return best_model.predict(
            np.asarray(x, dtype=np.float32),
            verbose=0
        ).ravel()

    kernel_explainer = shap.KernelExplainer(predict_fn, X_bg)
    shap_values = kernel_explainer.shap_values(X_explain, nsamples=100)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)
    X_explain_df = pd.DataFrame(
        X_explain,
        columns=transformed_feature_names
    )

    shap.summary_plot(
        shap_values,
        X_explain_df,
        show=False,
        max_display=20
    )
    plt.title("Global SHAP Summary — Neural Network")
    plt.tight_layout()
    plt.show()
elif DATA_READY:
    print("SHAP unavailable in this environment.")


## 14. Local SHAP explanation

In [ ]:
if DATA_READY and SHAP_AVAILABLE:
    local_i = 0
    local_values = shap_values[local_i]
    local_x = X_explain[local_i]
    local_base = float(
        np.asarray(kernel_explainer.expected_value).reshape(-1)[0]
    )

    local_exp = shap.Explanation(
        values=local_values,
        base_values=local_base,
        data=local_x,
        feature_names=transformed_feature_names,
    )

    probability = float(
        best_model.predict(
            local_x.reshape(1, -1),
            verbose=0
        ).ravel()[0]
    )

    print("Explained prediction probability:", round(probability, 4))

    shap.plots.waterfall(
        local_exp,
        max_display=15,
        show=False
    )
    plt.title("Local SHAP Explanation — One Driver/Race")
    plt.tight_layout()
    plt.show()


## 15. Inference function

In [ ]:
def predict_podium(record, model, preprocessor, raw_feature_cols, threshold):
    if isinstance(record, dict):
        row = pd.DataFrame([record])
    elif isinstance(record, pd.Series):
        row = record.to_frame().T
    elif isinstance(record, pd.DataFrame):
        row = record.copy()
    else:
        raise TypeError("record must be dict, Series, or DataFrame")

    for c in raw_feature_cols:
        if c not in row.columns:
            row[c] = np.nan

    row = row[raw_feature_cols]
    X_new = np.asarray(
        preprocessor.transform(row),
        dtype=np.float32
    )

    prob = float(
        model.predict(
            X_new,
            verbose=0
        ).ravel()[0]
    )

    return {
        "podium_probability": prob,
        "podium_prediction": int(prob >= threshold),
        "threshold": float(threshold),
    }

if DATA_READY:
    complete_record = X_test_raw.iloc[0].copy()

    print("Complete record prediction:")
    print(
        predict_podium(
            complete_record,
            best_model,
            preprocessor,
            feature_cols,
            best_threshold
        )
    )

    partially_missing_record = complete_record.copy()

    num_to_blank = max(1, min(3, len(feature_cols)))
    for c in feature_cols[:num_to_blank]:
        partially_missing_record[c] = np.nan

    print("\nPartially-missing record prediction:")
    print("Missing fields:", feature_cols[:num_to_blank])

    print(
        predict_podium(
            partially_missing_record,
            best_model,
            preprocessor,
            feature_cols,
            best_threshold
        )
    )


## 16. Final interpretation checklist

Before submission, report the actual executed values:

- best NN trial by **validation PR-AUC**
- selected threshold
- train / validation / test ROC-AUC, PR-AUC, F1
- most damaging feature-group removal in ablation
- top permutation-importance features
- global SHAP findings
- one local SHAP explanation
- statement that XAI importance is **not causal**

## Final integration check
The Kaggle notebook and GitHub version must use the same:
1. modeling table
2. leakage-safe features
3. preprocessing
4. chronological split
5. target definition
6. model metrics
